In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

steps = pd.read_csv("../data/walkforward_steps.csv", parse_dates=["train_end", "test_start", "test_end"])
equity = pd.read_csv("../data/walkforward_equity.csv", index_col=0, parse_dates=True).squeeze()

print("Per-step results:")
print(steps[["step", "test_start", "test_end", "hedge_ratio", "adf_statistic", "test_return", "test_sharpe"]].to_string(index=False))
print()
print(f"Stitched equity: {len(equity)} bars")
print(f"Range: {equity.index.min().date()} to {equity.index.max().date()}")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
equity.plot(ax=ax, color="navy", linewidth=1.3)
ax.axhline(equity.iloc[0], color="black", linestyle="--", alpha=0.4, label="Starting equity")

# Mark each step boundary
for _, row in steps.iterrows():
    ax.axvline(row["test_start"], color="gray", linestyle=":", alpha=0.5)
    # Annotate each step with its Sharpe
    mid_x = row["test_start"] + (row["test_end"] - row["test_start"]) / 2
    y_pos = equity.max() * 0.998
    color = "green" if row["test_sharpe"] > 0 else "red"
    ax.text(mid_x, y_pos, f"S{int(row['step'])}\nSh={row['test_sharpe']:.2f}",
            ha="center", va="top", fontsize=9, color=color, fontweight="bold")

ax.set_title("NDSN/OTIS walk-forward equity (stitched, all out-of-sample)")
ax.set_ylabel("Equity ($)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 8), sharex=True)

axes[0].plot(steps["test_start"], steps["hedge_ratio"], "o-", color="navy", linewidth=1.5)
axes[0].set_title("Hedge ratio (β) drift across walk-forward steps")
axes[0].set_ylabel("β")
axes[0].grid(alpha=0.3)
axes[0].axhline(0.65, color="black", linestyle=":", alpha=0.4, label="Full-window β = 0.65")
axes[0].legend()

axes[1].plot(steps["test_start"], steps["adf_statistic"], "o-", color="crimson", linewidth=1.5)
axes[1].axhline(-3.37, color="red", linestyle="--", alpha=0.5, label="EG 5% critical value")
axes[1].set_title("ADF statistic on residuals (more negative = stronger cointegration)")
axes[1].set_ylabel("ADF stat")
axes[1].grid(alpha=0.3)
axes[1].legend()

axes[2].bar(range(len(steps)), steps["test_sharpe"],
            color=["green" if s > 0 else "red" for s in steps["test_sharpe"]])
axes[2].axhline(0, color="black", linewidth=0.7)
axes[2].set_title("Per-step out-of-sample Sharpe")
axes[2].set_ylabel("Sharpe")
axes[2].set_xlabel("Step")
axes[2].set_xticks(range(len(steps)))
axes[2].set_xticklabels([f"S{int(s)}\n{d.date()}" for s, d in zip(steps["step"], steps["test_start"])])
axes[2].grid(alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [ ]:
# Simulate: trade only steps with trailing ADF < -3.37 (the 5% threshold)
threshold = -3.37
traded_steps = steps[steps["adf_statistic"] < threshold]
skipped_steps = steps[steps["adf_statistic"] >= threshold]

print(f"With ADF gating at {threshold}:")
print(f"  Steps traded:  {len(traded_steps)} of {len(steps)}")
print(f"  Steps skipped: {len(skipped_steps)}")
print()

if len(traded_steps) > 0:
    print(f"Traded steps:")
    print(traded_steps[["step", "test_start", "adf_statistic", "test_return", "test_sharpe"]].to_string(index=False))

if len(skipped_steps) > 0:
    print(f"\nSkipped steps (what we'd have avoided):")
    print(skipped_steps[["step", "test_start", "adf_statistic", "test_return", "test_sharpe"]].to_string(index=False))

# Compute aggregate Sharpe of traded-only periods
# (approximation: average per-step Sharpes weighted by number of trading days)
if len(traded_steps) > 0:
    weights = (traded_steps["test_end"] - traded_steps["test_start"]).dt.days
    weighted_sharpe = (traded_steps["test_sharpe"] * weights).sum() / weights.sum()
    print(f"\nApprox weighted Sharpe of traded-only periods: {weighted_sharpe:.2f}")